# 01 · SFT через LoRA

**Цель:** модель матерится, когда речь о выпускной работе, и отвечает нормально обо всём остальном. Поведение утрированное намеренно: его видно без метрик, а контрольная группа проверяет главное — выучилось ли правило «когда», а не только «как».

**Схема:** замер до → адаптер → обучение → замер после на тех же шести запросах. Математика — `books/02-sft-math.pdf`. Все вызовы `peft` и `transformers` на виду.

In [ ]:
from common import MODEL_ID, SYSTEM, RUNS, read_raw, swears, demo_answers, show, side_by_side, swear_suite, fmt

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from vlmkit import Sample, ChatCollator, memory_report, preview, evaluate as ev
from vlmkit.compat import supported, first_accepted

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## До

In [ ]:
suite = swear_suite()

before = demo_answers(model, processor)
before_metrics = ev.run(model, processor, suite)

show(before, "ДО ОБУЧЕНИЯ", detector=swears)
print("\nметрики:", fmt(before_metrics))

## Данные

Чётные строки — в обучение, нечётные остаются на замер. Целевой ответ выбирает группа: о дипломе — с матом, об остальном — обычный. Без второй половины модель выучит «материться всегда»; на это и смотрит колонка «ложные».

In [ ]:
rows = read_raw("swear.jsonl")[::2]
target = lambda r: r["swear"] if r["group"] == "topic" else r["plain"]
train = [Sample.from_qa(r["question"], target(r)) for r in rows]
held = [Sample.from_qa(r["question"], target(r)) for r in read_raw("swear.jsonl")[1::2]]

print(len(train), "примеров, из них с матом", sum(r["group"] == "topic" for r in rows))
print(preview(train[0], processor, system=SYSTEM))

ppl_before = ev.perplexity(model, processor, held, system=SYSTEM)
print(f"\nperplexity целевых ответов на отложенной половине: {ppl_before:.1f}")

## Адаптер

Три решения в конфиге, которые не косметика:

- **регекс с отрицательным просмотром** выкидывает визуальную башню — переучивать её на паре десятков текстов вредно;
- **`use_rslora=True`** меняет масштаб с `α/r` на `α/√r`, иначе вклад адаптера затухает с ростом ранга;
- **`r=16`** — для стиля ответа достаточно: это изменение манеры, не новый навык.

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora)
model.print_trainable_parameters()

# Без этого при gradient checkpointing градиент не доходит до адаптеров.
# Ошибки не будет — обучение просто ничего не даст.
model.enable_input_require_grads()
model.config.use_cache = False

## Обучение

`remove_unused_columns=False` критично: иначе `Trainer` выбросит из батча всё, чего нет в сигнатуре `forward`. `supported()` отсеивает аргументы, которых нет в вашей версии `transformers`.

In [ ]:
collator = ChatCollator(processor, system=SYSTEM)

args = dict(
    output_dir=str(RUNS / "sft-swear"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-4,          # для LoRA в 100 раз выше полного FT — это нормально
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused",
    logging_steps=5,
    save_strategy="no",
    remove_unused_columns=False,
    report_to=[],
    seed=42,
)
args |= first_accepted(TrainingArguments, {"warmup_ratio": 0.05, "warmup_steps": 2})

trainer = Trainer(
    model=model,
    args=TrainingArguments(**supported(TrainingArguments, args)),
    train_dataset=train,
    data_collator=collator,
)
result = trainer.train()
print(f"loss {result.training_loss:.3f}, {result.metrics.get('train_runtime', 0)/60:.1f} мин")

## После

Те же шесть запросов. ✓ — в ответе есть мат. Правильная картина: ✓ на трёх верхних (`topic`), · на трёх нижних (`other`).

In [ ]:
model.eval()
after = demo_answers(model, processor)
after_metrics = ev.run(model, processor, suite)

show(after, "ПОСЛЕ ОБУЧЕНИЯ", detector=swears)
side_by_side(before, after, detector=swears)

print(f"\nдо:    {fmt(before_metrics)}")
print(f"после: {fmt(after_metrics)}")

ppl_after = ev.perplexity(model, processor, held, system=SYSTEM)
print(f"perplexity целевых ответов: {ppl_before:.1f} → {ppl_after:.1f}")

## Адаптер как переключатель

Исходные веса не менялись. Адаптер можно выключить на лету — один процесс отвечает и так, и так.

In [ ]:
with model.disable_adapter():
    print("адаптер выключен:", fmt(ev.run(model, processor, suite)))
print("адаптер включён: ", fmt(ev.run(model, processor, suite)))

model.save_pretrained(str(RUNS / "sft-swear"))

## На что смотреть

**Попадание выросло, ложные тоже** — переобобщение: выучилось «как», но не «когда». Больше примеров группы `other`, меньше эпох.

**Попадание не выросло** — смотрите сырые ответы, не метрику. Если ответ не в ⟦скобках⟧ в `preview`, градиента нет. Если модель отвечает по-английски или уходит в рассуждение — детектор честно ставит ·, но причина не в обучении.

**Ответы стали обрывочными** — скорость обучения велика, попробуйте 5e-5.